# Section 5.6 - Sensitivity Analysis

This notebook builds the paper-ready figures for the sensitivity analysis section.

Purpose:
- test whether the main conclusions hold across different conditions

The section uses only three dimensions:
1. Estimation window
2. Universe
3. Construction date

The figure set is intentionally minimal:
- `fig:window_sensitivity`
- `fig:universe_sensitivity`
- `fig:date_sensitivity`


In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

PROJECT_ROOT = Path.cwd()
OUTPUT_CANDIDATES = [
    PROJECT_ROOT / 'outputs' / 'data_exports' / 'final_experimental_setup',
    PROJECT_ROOT / 'outputs' / 'final_experimental_setup',
]
OUTPUT_DIR = next((path for path in OUTPUT_CANDIDATES if path.exists()), OUTPUT_CANDIDATES[0])
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'paper_figures' / 'section_5_6'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

bt = pd.read_csv(TABLE_DIR / 'backtest_summary.csv')
print(f'Using output directory: {OUTPUT_DIR}')
print(f'Backtest rows: {len(bt)}')
print(f'Figures will be written to: {FIGURE_DIR}')


Using output directory: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup
Backtest rows: 135
Figures will be written to: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup\paper_figures\section_5_6


In [2]:
METHOD_ORDER = [
    'equal_weight',
    'naive_risk_parity',
    'markowitz',
    'hrp_recursive',
    'hca_deprado_ew_nrp',
]

PALETTE = {
    'equal_weight': '#4C78A8',
    'naive_risk_parity': '#72B7B2',
    'markowitz': '#F58518',
    'hrp_recursive': '#54A24B',
    'hca_deprado_ew_nrp': '#E45756',
}


def method_label(method: str) -> str:
    labels = {
        'equal_weight': 'Equal Weight (1/N)',
        'naive_risk_parity': 'Naive Risk Parity',
        'markowitz': 'Markowitz',
        'hrp_recursive': 'HRP Recursive',
        'hca_deprado_ew_nrp': 'HCA De Prado EW/NRP',
    }
    return labels.get(method, method)


def universe_label(universe: str) -> str:
    labels = {
        'djia': 'DJIA',
        'multi_asset_etf': 'Multi-Asset ETF',
        'sp100_style_top100': 'S&P 100-style Top 100',
    }
    return labels.get(universe, universe)


def apply_paper_layout(fig: go.Figure, *, title: str, width: int = 1100, height: int = 620) -> None:
    fig.update_layout(
        title=title,
        template='plotly_white',
        width=width,
        height=height,
        font=dict(size=13),
        margin=dict(l=70, r=40, t=90, b=65),
        legend=dict(orientation='h', x=0, y=1.02, xanchor='left', yanchor='bottom'),
        paper_bgcolor='white',
        plot_bgcolor='white',
        barmode='group',
    )


def save_figure(fig: go.Figure, filename: str) -> None:
    path = FIGURE_DIR / filename
    pio.write_html(fig, file=path, include_plotlyjs='cdn', full_html=True)
    print(f'Saved: {path}')


## Figure 1 - Sharpe by Estimation Window

This figure answers: does performance depend on the size of the estimation window?


In [3]:
window_df = (
    bt.groupby(['estimation_months', 'method'], as_index=False)['sharpe_ratio']
    .mean()
)
window_df['method_label'] = pd.Categorical(
    window_df['method'].map(method_label),
    categories=[method_label(m) for m in METHOD_ORDER],
    ordered=True,
)
window_df = window_df.sort_values(['estimation_months', 'method_label'])
window_df['window_label'] = window_df['estimation_months'].astype(str) + ' months'

fig_window = px.bar(
    window_df,
    x='window_label',
    y='sharpe_ratio',
    color='method_label',
    text='sharpe_ratio',
    category_orders={
        'window_label': ['12 months', '24 months', '36 months'],
        'method_label': [method_label(m) for m in METHOD_ORDER],
    },
    color_discrete_map={method_label(k): v for k, v in PALETTE.items()},
)
fig_window.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig_window.update_xaxes(title='Estimation window')
fig_window.update_yaxes(title='Average Sharpe ratio')
apply_paper_layout(fig_window, title='Sensitivity of Sharpe Ratio to Estimation Window')
save_figure(fig_window, 'figure_01_window_sensitivity.html')
fig_window


Saved: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup\paper_figures\section_5_6\figure_01_window_sensitivity.html


## Figure 2 - Sharpe by Universe

This figure answers: do the methods behave differently across universes?


In [4]:
universe_df = (
    bt.groupby(['universe_id', 'method'], as_index=False)['sharpe_ratio']
    .mean()
)
universe_df['universe_label'] = pd.Categorical(
    universe_df['universe_id'].map(universe_label),
    categories=['DJIA', 'S&P 100-style Top 100', 'Multi-Asset ETF'],
    ordered=True,
)
universe_df['method_label'] = pd.Categorical(
    universe_df['method'].map(method_label),
    categories=[method_label(m) for m in METHOD_ORDER],
    ordered=True,
)
universe_df = universe_df.sort_values(['universe_label', 'method_label'])

fig_universe = px.bar(
    universe_df,
    x='universe_label',
    y='sharpe_ratio',
    color='method_label',
    text='sharpe_ratio',
    category_orders={
        'universe_label': ['DJIA', 'S&P 100-style Top 100', 'Multi-Asset ETF'],
        'method_label': [method_label(m) for m in METHOD_ORDER],
    },
    color_discrete_map={method_label(k): v for k, v in PALETTE.items()},
)
fig_universe.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig_universe.update_xaxes(title='Universe')
fig_universe.update_yaxes(title='Average Sharpe ratio')
apply_paper_layout(fig_universe, title='Sensitivity of Sharpe Ratio to Universe', width=1180)
save_figure(fig_universe, 'figure_02_universe_sensitivity.html')
fig_universe


Saved: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup\paper_figures\section_5_6\figure_02_universe_sensitivity.html


## Figure 3 - Sharpe by Construction Date

This figure answers: do the methods change across market regimes?


In [5]:
date_df = (
    bt.groupby(['construction_date', 'method'], as_index=False)['sharpe_ratio']
    .mean()
)
date_df['date_label'] = pd.Categorical(
    pd.to_datetime(date_df['construction_date']).dt.strftime('%Y-%m-%d'),
    categories=['2019-06-01', '2020-06-01', '2022-01-03'],
    ordered=True,
)
date_df['method_label'] = pd.Categorical(
    date_df['method'].map(method_label),
    categories=[method_label(m) for m in METHOD_ORDER],
    ordered=True,
)
date_df = date_df.sort_values(['date_label', 'method_label'])

fig_date = px.bar(
    date_df,
    x='date_label',
    y='sharpe_ratio',
    color='method_label',
    text='sharpe_ratio',
    category_orders={
        'date_label': ['2019-06-01', '2020-06-01', '2022-01-03'],
        'method_label': [method_label(m) for m in METHOD_ORDER],
    },
    color_discrete_map={method_label(k): v for k, v in PALETTE.items()},
)
fig_date.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig_date.update_xaxes(title='Construction date')
fig_date.update_yaxes(title='Average Sharpe ratio')
apply_paper_layout(fig_date, title='Sensitivity of Sharpe Ratio to Market Regime')
save_figure(fig_date, 'figure_03_date_sensitivity.html')
fig_date


Saved: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup\paper_figures\section_5_6\figure_03_date_sensitivity.html


## Writing Prompts

Use this flow in the paper:

1. Start with estimation-window sensitivity.
2. Then discuss universe sensitivity.
3. Then discuss regime sensitivity.
4. End by stating whether the main conclusions remain robust.

One-line memory:

`5.6 = do results hold everywhere?`
